# 13 — Re-eval only the embedding-dependent models

After the multilingual re-embed (notebook 12), only the models that consume `embeddings.npy`
change. This notebook re-scores **just those**, on the **same split / users / protocols as
`08_evaluation`** so the numbers are directly comparable, and diffs them against the old
(bge-small-en) results.

**Changed → re-evaluated here:** `content_emb`, `maxsim`, `similar` (UC4), and the learned
hybrids that use `content_emb` (`hybrid_cf_content`, `…_pop`, `…_popneg`).

**Unchanged → not re-run** (no embedding input; they reproduce their old numbers): `popularity`,
`svd`, `content_tfidf_full`, `content_bow`, Mult-VAE. The study **headline (Mult-VAE ⊕ TF-IDF
max-sim) does not move** — it never touches these embeddings.

Expect `content_emb` NDCG@10 to move only a little: the held-out test is the same mostly-English
interactions, so a multilingual encoder mostly changes *which* English neighbours rank, not the
accuracy ceiling. The multilingual win (no foreign-seed hijack) is a serving-robustness property
this aggregate metric doesn't probe.

## Kaggle bootstrap
Pulls the package (master) + links inputs into `artifacts/`. No-op off Kaggle.

In [ ]:
import os, sys, glob
if os.path.exists('/kaggle'):
    REPO = '/kaggle/working/book_recsys'
    if not os.path.exists(REPO):
        !git clone -b master https://github.com/MayaDeneva/book_recsys.git {REPO}
    else:
        !cd {REPO} && git pull origin master
    !pip install -e {REPO} -q
    sys.path.insert(0, REPO)
    os.chdir(REPO)
    os.makedirs('artifacts', exist_ok=True)
    for f in ['sample.parquet', 'catalog.parquet', 'embeddings.npy', 'models.joblib']:
        src = glob.glob(f'/kaggle/input/**/{f}', recursive=True)
        if src and not os.path.exists(f'artifacts/{f}'):
            os.symlink(src[0], f'artifacts/{f}')
    print('Kaggle: artifacts linked:', sorted(os.listdir('artifacts')))
else:
    print('not on Kaggle — running from local repo, expecting artifacts/ in place')

## Setup — same split, users, and harness as notebook 08

In [2]:
import glob, os, joblib, numpy as np, pandas as pd
from book_recsys.data.splits import leave_last_n_out
from book_recsys.eval.harness import (build_user_histories, build_relevance, evaluate_per_user,
                                       evaluate_sampled_negatives, evaluate_similar,
                                       build_cooccurrence_relevance)
from book_recsys.models.content.maxsim import MaxSimRecommender

def find(name):
    for p in (f'artifacts/{name}', f'../artifacts/{name}'):
        if glob.glob(p): return glob.glob(p)[0]
    raise FileNotFoundError(name)
REPORTS = next((p for p in ('reports', '../reports') if os.path.isdir(p)), 'reports')
os.makedirs(REPORTS, exist_ok=True)

sample  = pd.read_parquet(find('sample.parquet'))
catalog = pd.read_parquet(find('catalog.parquet'))
book_ids = catalog['book_id'].tolist()

N_USERS = 3000                                  # match 08 for comparable CIs
train, holdout = leave_last_n_out(sample, n=1)  # same split 08 holds out from
rng = np.random.default_rng(0)                  # same seed -> same test users as 08
test_users = rng.choice(holdout['user_id'].unique(), size=N_USERS, replace=False)
histories = build_user_histories(train[train['user_id'].isin(test_users)])
relevance = build_relevance(holdout[holdout['user_id'].isin(test_users)])
print(len(relevance), 'test users · catalog', catalog.shape)

3000 test users · catalog (468628, 9)


In [4]:
# Load the new (re-embedded) zoo; rebuild maxsim from the new embeddings.npy.
models = joblib.load(find('models.joblib'))
models.setdefault('maxsim', MaxSimRecommender(book_ids, np.load(find('embeddings.npy'))).fit())
# sklearn cross-version compat for the learned hybrids' LogisticRegression (as in 08).
from sklearn.linear_model import LogisticRegression
for _m in models.values():
    for _, _est in getattr(getattr(_m, '_model', None), 'steps', None) or []:
        if isinstance(_est, LogisticRegression):
            _est.multi_class = 'auto'

CHANGED_FULL    = ['content_emb']                                  # full-catalog (08 protocol)
CHANGED_SAMPLED = ['content_emb', 'maxsim', 'hybrid_cf_content',
                   'hybrid_cf_content_pop', 'hybrid_cf_content_popneg']
print('re-evaluating:', sorted(set(CHANGED_FULL + CHANGED_SAMPLED) | {'similar (UC4)'}))

/opt/anaconda3/envs/book_recsys/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/book_recsys/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/book_recsys/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.6.1 w

re-evaluating: ['content_emb', 'hybrid_cf_content', 'hybrid_cf_content_pop', 'hybrid_cf_content_popneg', 'maxsim', 'similar (UC4)']


## 1. Full-catalog leave-1-out (content_emb) — optional, ~3-5 min
Skip this cell if you only want the (fast) sampled comparison below; the diff cell
handles a missing full-catalog run.


In [3]:
full = {}
for name in CHANGED_FULL:
    pu = evaluate_per_user(models[name], histories, relevance, k=10)
    full[name] = {m: float(np.mean(v)) for m, v in pu.items()}
    print(name, full[name])
full_df = pd.DataFrame(full).T[['recall@10', 'ndcg@10', 'mrr']].round(4)
full_df.to_csv(f'{REPORTS}/study_results_reembed.csv')
display(full_df)

NameError: name 'CHANGED_FULL' is not defined

## 2. Sampled negatives — uniform + popularity-weighted (08 HEADLINE protocol)

In [5]:
counts = train['book_id'].value_counts()
item_weights = counts.reindex(book_ids).fillna(0).to_numpy()   # aligned to book_ids
uni, pw = {}, {}
for name in CHANGED_SAMPLED:
    uni[name] = evaluate_sampled_negatives(models[name], histories, relevance, book_ids,
                                           n_neg=100, k=10, seed=0)
    pw[name]  = evaluate_sampled_negatives(models[name], histories, relevance, book_ids,
                                           n_neg=100, k=10, seed=0, item_weights=item_weights)
    print(f'{name:26} uniform {uni[name]}  ·  popw {pw[name]}')
uni_df = pd.DataFrame(uni).T.sort_values('ndcg@10', ascending=False).round(4)
pw_df  = pd.DataFrame(pw).T.sort_values('ndcg@10', ascending=False).round(4)
uni_df.to_csv(f'{REPORTS}/study_sampled_reembed.csv')
pw_df.to_csv(f'{REPORTS}/study_sampled_popneg_reembed.csv')
print('\nuniform negatives:'); display(uni_df)
print('popularity-weighted negatives:'); display(pw_df)

content_emb                uniform {'recall@10': 0.21533333333333332, 'ndcg@10': 0.11862339196425724, 'mrr': 0.11393146212123567}  ·  popw {'recall@10': 0.21, 'ndcg@10': 0.12105931562804637, 'mrr': 0.11722189260321987}
maxsim                     uniform {'recall@10': 0.347, 'ndcg@10': 0.23029355626796283, 'mrr': 0.21506808181753145}  ·  popw {'recall@10': 0.335, 'ndcg@10': 0.22116555694669998, 'mrr': 0.20610708504196548}
hybrid_cf_content          uniform {'recall@10': 0.6776666666666666, 'ndcg@10': 0.5694781270928262, 'mrr': 0.5449405699916788}  ·  popw {'recall@10': 0.373, 'ndcg@10': 0.19973382210290064, 'mrr': 0.17237005759732985}
hybrid_cf_content_pop      uniform {'recall@10': 0.7546666666666667, 'ndcg@10': 0.6111628518035805, 'mrr': 0.5756603283746666}  ·  popw {'recall@10': 0.36133333333333334, 'ndcg@10': 0.19002296353808792, 'mrr': 0.16025494083952277}
hybrid_cf_content_popneg   uniform {'recall@10': 0.41, 'ndcg@10': 0.3029907314375802, 'mrr': 0.2893404821928937}  ·  popw {'rec

,recall@10,ndcg@10,mrr
hybrid_cf_content_pop,0.7547,0.6112,0.5757
hybrid_cf_content,0.6777,0.5695,0.5449
hybrid_cf_content_popneg,0.4100,0.3030,0.2893
maxsim,0.3470,0.2303,0.2151
content_emb,0.2153,0.1186,0.1139


popularity-weighted negatives:


,recall@10,ndcg@10,mrr
maxsim,0.3350,0.2212,0.2061
hybrid_cf_content,0.3730,0.1997,0.1724
hybrid_cf_content_popneg,0.3427,0.1979,0.1738
hybrid_cf_content_pop,0.3613,0.1900,0.1603
content_emb,0.2100,0.1211,0.1172


## 3. UC4 — similar-to-anchor (content neighbours vs co-read)

In [7]:
anchors = list(np.random.default_rng(0).choice(
    counts[counts >= 50].index.tolist(), 500, replace=False))   # same anchors as 08
uc4_rel = build_cooccurrence_relevance(sample, anchors, top_n=10)
uc4 = evaluate_similar(models['similar'], uc4_rel, k=10)
pd.DataFrame([uc4]).round(4).to_csv(f'{REPORTS}/study_uc4_reembed.csv', index=False)
print('UC4 similar (re-embed) vs co-read:', uc4)

KeyboardInterrupt: 

## 4. Old (bge-small-en) vs new (multilingual) — did the swap cost accuracy?

In [8]:
# Old vs new NDCG@10 for the changed models. _old(fname) loads a previously-committed results CSV
# (model-name index) from reports/ and returns its ndcg@10 column, or None if the file is absent.
def _old(path, col='ndcg@10'):
    p = next((q for q in (f'{REPORTS}/{path}', path) if os.path.exists(q)), None)
    return pd.read_csv(p, index_col=0)[col] if p else None

rows = []
# Full-catalog block — only if Section 1 was actually run (skips cleanly if you ran sampled-only).
old_full = _old('study_results.csv')
if old_full is not None and 'full' in globals():
    for name in CHANGED_FULL:
        if name in old_full.index and name in full:
            rows.append({'model': name, 'protocol': 'full-cat', 'old': round(float(old_full[name]), 4),
                         'new': full[name]['ndcg@10'],
                         'delta': round(full[name]['ndcg@10'] - float(old_full[name]), 4)})
# Sampled negatives, uniform.
old_uni = _old('study_sampled.csv')
if old_uni is not None:
    for name in CHANGED_SAMPLED:
        if name in old_uni.index:
            rows.append({'model': name, 'protocol': 'sampled-uni', 'old': round(float(old_uni[name]), 4),
                         'new': uni[name]['ndcg@10'],
                         'delta': round(uni[name]['ndcg@10'] - float(old_uni[name]), 4)})
# Sampled negatives, popularity-weighted — the study's HEADLINE protocol.
old_pw = _old('study_sampled_popneg.csv')
if old_pw is not None:
    for name in CHANGED_SAMPLED:
        if name in old_pw.index:
            rows.append({'model': name, 'protocol': 'sampled-popw', 'old': round(float(old_pw[name]), 4),
                         'new': pw[name]['ndcg@10'],
                         'delta': round(pw[name]['ndcg@10'] - float(old_pw[name]), 4)})
if rows:
    diff = pd.DataFrame(rows)
    print('NDCG@10 - bge-small-en (old) vs multilingual MiniLM (new):'); display(diff)
    diff.to_csv(f'{REPORTS}/study_reembed_diff.csv', index=False)
else:
    print('No committed old CSVs found to diff against - new numbers saved with the _reembed suffix.')

KeyError: 'content_emb'

**Reading the diff:** small negative deltas on the English-heavy test are expected and acceptable
— the encoder changed to gain cross-language robustness, not English accuracy. A *large* drop on
`content_emb` would mean the multilingual MiniLM is too weak on English; if so, re-embed with a
stronger multilingual model (`BAAI/bge-m3` or `intfloat/multilingual-e5-large`) and re-run this
notebook. The headline model is unaffected either way.